# Практична робота: моделювання в озері даних

Медальйонна архітектура **Bronze → Silver → Gold** для даних мережі кав’ярень.

In [4]:
# Якщо бібліотеки ще не встановлені, виконайте цю комірку один раз
%pip install -q pandas pyarrow azure-storage-blob


Note: you may need to restart the kernel to use updated packages.


## 1. Підключення


In [ ]:
from azure.storage.blob import ContainerClient
from io import BytesIO
import pandas as pd

# Вставте повні SAS-посилання з навчальних матеріалів
SOURCE_SAS = "https://datalakedatalab.blob.core.windows.net/data-lake?sp=r&st=..."

LAKE_SAS = "https://datalakedatalab.blob.core.windows.net/students-homework?sp=racwl&st=..."

source = ContainerClient.from_container_url(SOURCE_SAS)
lake = ContainerClient.from_container_url(LAKE_SAS)

# Власний префікс: латиницею, малими літерами, без пробілів
STUDENT = "svierchkova_svitlana"


def read_json(name):
    raw = source.download_blob(name).readall()
    return pd.read_json(BytesIO(raw))


def read_parquet(path):
    raw = lake.download_blob(f"{STUDENT}/{path}").readall()
    return pd.read_parquet(BytesIO(raw))


def write_parquet(df, path):
    buf = BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    lake.upload_blob(
        name=f"{STUDENT}/{path}",
        data=buf.getvalue(),
        overwrite=True
    )

print("Підключення налаштовано. STUDENT:", STUDENT)


Підключення налаштовано. STUDENT: svierchkova_svitlana


In [ ]:
shops = read_json("shops.json")
products = read_json("products.json")

print("shops.json:", len(shops))       # очікувано 5
print("products.json:", len(products)) # очікувано 12

assert len(shops) == 5
assert len(products) == 12


## 2. Bronze: сирі дані без змін


In [ ]:
parts = [read_json(f"orders_2026_0{m}.json") for m in (1, 2, 3)]
bronze_orders = pd.concat(parts, ignore_index=True)

print("Bronze orders rows:", len(bronze_orders)) # очікувано 980
assert len(bronze_orders) == 980


In [ ]:
write_parquet(bronze_orders, "bronze/orders.parquet")
write_parquet(shops, "bronze/shops.parquet")
write_parquet(products, "bronze/products.parquet")

print("Bronze записано")


In [ ]:
bronze_files = []
for blob in lake.list_blobs(name_starts_with=f"{STUDENT}/bronze/"):
    bronze_files.append(blob.name)
    print(blob.name, blob.size)

assert len(bronze_files) == 3


## 3. Silver: очищення і типізація


In [ ]:
bronze_orders = read_parquet("bronze/orders.parquet")

print("Повні дублікати:", bronze_orders.duplicated().sum())
print("Унікальні shop_code:", bronze_orders["shop_code"].nunique(dropna=True))
print("Унікальні payment_type:", bronze_orders["payment_type"].nunique(dropna=True))
print("Порожні qty:", bronze_orders["qty"].isna().sum())

assert bronze_orders.duplicated().sum() == 20
assert bronze_orders["shop_code"].nunique(dropna=True) == 10
assert bronze_orders["payment_type"].nunique(dropna=True) == 4
assert bronze_orders["qty"].isna().sum() == 36


In [ ]:
silver = bronze_orders.drop_duplicates().copy()

silver["shop_code"] = silver["shop_code"].str.strip().str.upper()
silver["payment_type"] = silver["payment_type"].str.strip().str.lower().map({
    "card": "Card",
    "cash": "Cash"
})
silver["qty"] = silver["qty"].fillna(1).astype(int)
silver["order_ts"] = pd.to_datetime(silver["order_ts"], errors="raise")
silver["order_date"] = silver["order_ts"].dt.date
silver["amount"] = silver["qty"] * silver["unit_price"]

print("Silver rows:", len(silver))
print("shop_code:", sorted(silver["shop_code"].unique()))
print("payment_type:", sorted(silver["payment_type"].dropna().unique()))
print("Порожні qty:", silver["qty"].isna().sum())

assert len(silver) == 960
assert set(silver["shop_code"].unique()) == {"S01", "S02", "S03", "S04", "S05"}
assert set(silver["payment_type"].dropna().unique()) == {"Card", "Cash"}
assert silver["payment_type"].notna().all()
assert silver["qty"].isna().sum() == 0


In [ ]:
write_parquet(silver, "silver/orders.parquet")

# Silver читає довідники тільки з Bronze
silver_shops = read_parquet("bronze/shops.parquet")
silver_products = read_parquet("bronze/products.parquet")

write_parquet(silver_shops, "silver/shops.parquet")
write_parquet(silver_products, "silver/products.parquet")

print("Silver записано")


## 4. Gold: зоряна схема


In [ ]:
# Gold читає тільки із Silver
silver_orders = read_parquet("silver/orders.parquet")
silver_shops = read_parquet("silver/shops.parquet")
silver_products = read_parquet("silver/products.parquet")

# Вимір кав'ярень
dim_shop = silver_shops.sort_values("shop_code").reset_index(drop=True)
dim_shop.insert(0, "shop_key", range(1, len(dim_shop) + 1))

# Вимір продуктів
dim_product = silver_products.sort_values("product_code").reset_index(drop=True)
dim_product.insert(0, "product_key", range(1, len(dim_product) + 1))

print("dim_shop rows:", len(dim_shop))
print("dim_product rows:", len(dim_product))


In [ ]:
fact_sales = (
    silver_orders
    .merge(
        dim_shop[["shop_key", "shop_code"]],
        on="shop_code",
        how="left",
        validate="many_to_one"
    )
    .merge(
        dim_product[["product_key", "product_code"]],
        on="product_code",
        how="left",
        validate="many_to_one"
    )
)

assert fact_sales["shop_key"].notna().all()
assert fact_sales["product_key"].notna().all()

fact_sales["shop_key"] = fact_sales["shop_key"].astype(int)
fact_sales["product_key"] = fact_sales["product_key"].astype(int)

fact_sales = fact_sales[[
    "order_id",
    "shop_key",
    "product_key",
    "order_date",
    "qty",
    "unit_price",
    "amount",
    "payment_type"
]]

print("fact_sales rows:", len(fact_sales))
fact_sales.head()


In [ ]:
fact_daily_sales = (
    fact_sales
    .groupby(["shop_key", "order_date"], as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        items_sold=("qty", "sum"),
        revenue=("amount", "sum")
    )
    .sort_values(["shop_key", "order_date"])
    .reset_index(drop=True)
)

print("fact_daily_sales rows:", len(fact_daily_sales))
fact_daily_sales.head()


In [ ]:
write_parquet(dim_shop, "gold/dim_shop.parquet")
write_parquet(dim_product, "gold/dim_product.parquet")
write_parquet(fact_sales, "gold/fact_sales.parquet")
write_parquet(fact_daily_sales, "gold/fact_daily_sales.parquet")

print("Gold записано")


## 5. Перевірка та аналітика


In [ ]:
dim_shop = read_parquet("gold/dim_shop.parquet")
dim_product = read_parquet("gold/dim_product.parquet")
fact_sales = read_parquet("gold/fact_sales.parquet")
fact_daily_sales = read_parquet("gold/fact_daily_sales.parquet")

print("dim_shop:", len(dim_shop))
print("dim_product:", len(dim_product))
print("fact_sales:", len(fact_sales))
print("fact_daily_sales:", len(fact_daily_sales))

assert len(dim_shop) == 5
assert len(dim_product) == 12
assert len(fact_sales) == 960
assert len(fact_daily_sales) == 396


In [ ]:
shop_keys_valid = fact_sales["shop_key"].isin(dim_shop["shop_key"]).all()
product_keys_valid = fact_sales["product_key"].isin(dim_product["product_key"]).all()
revenue_equal = fact_sales["amount"].sum() == fact_daily_sales["revenue"].sum()

print(shop_keys_valid)
print(product_keys_valid)
print(revenue_equal)

assert shop_keys_valid
assert product_keys_valid
assert revenue_equal


In [ ]:
revenue_by_category = (
    fact_sales
    .merge(
        dim_product[["product_key", "category"]],
        on="product_key",
        how="left",
        validate="many_to_one"
    )
    .groupby("category", as_index=False)
    .agg(revenue=("amount", "sum"))
    .sort_values("revenue", ascending=False)
    .reset_index(drop=True)
)

print(revenue_by_category.to_string(index=False))

expected_category_revenue = {
    "Coffee": 49915,
    "Bakery": 19955,
    "Dessert": 14850,
    "Tea": 5600
}
actual_category_revenue = dict(zip(
    revenue_by_category["category"],
    revenue_by_category["revenue"]
))
assert actual_category_revenue == expected_category_revenue


In [ ]:
revenue_by_shop = (
    fact_sales
    .merge(
        dim_shop[["shop_key", "shop_name"]],
        on="shop_key",
        how="left",
        validate="many_to_one"
    )
    .groupby("shop_name", as_index=False)
    .agg(revenue=("amount", "sum"))
    .sort_values("revenue", ascending=False)
    .reset_index(drop=True)
)

print(revenue_by_shop.to_string(index=False))

assert revenue_by_shop.iloc[0]["shop_name"] == "Rynok Coffee"
assert revenue_by_shop.iloc[0]["revenue"] == 20075


## Фінальна перевірка структури озера


In [ ]:
all_files = []
for blob in lake.list_blobs(name_starts_with=f"{STUDENT}/"):
    all_files.append(blob.name)
    print(blob.name, blob.size)

expected_files = {
    f"{STUDENT}/bronze/orders.parquet",
    f"{STUDENT}/bronze/shops.parquet",
    f"{STUDENT}/bronze/products.parquet",
    f"{STUDENT}/silver/orders.parquet",
    f"{STUDENT}/silver/shops.parquet",
    f"{STUDENT}/silver/products.parquet",
    f"{STUDENT}/gold/dim_shop.parquet",
    f"{STUDENT}/gold/dim_product.parquet",
    f"{STUDENT}/gold/fact_sales.parquet",
    f"{STUDENT}/gold/fact_daily_sales.parquet"
}

actual_files = set(all_files)
print("Кількість файлів:", len(actual_files))
print("Очікувана структура створена:", expected_files.issubset(actual_files))

assert len(actual_files) == 10
assert actual_files == expected_files
print("Усі перевірки пройдено успішно")
